## 334 - Accessing and Plotting NEXRAD Level 3 Data

[Youtube](https://www.youtube.com/watch?v=NPgIX10arHw)

In [1]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime, timedelta
from pathlib import Path
from metpy.calc import azimuth_range_to_lat_lon
from metpy.io import Level3File
from metpy.plots import USCOUNTIES, add_timestamp, colortables
from metpy.remote import NEXRADLevel3Archive
from metpy.units import units

In [2]:
site = 'TLX'
product = 'N0B'
start = datetime(2023, 4, 20, 0, 0)
end = start + timedelta(hours=2)

products = list(NEXRADLevel3Archive().get_range(site, product, start, end))
print(f'Found {len(products)} products')

Found 73 products


In [3]:
outdir = Path('level3images')
outdir.mkdir(exist_ok=True)

In [4]:
extent = [-97.95, -97.0, 34.85, 35.65]
norm, cmap = colortables.get_with_steps('NWSStormClearReflectivity', -20, 0.5)

In [5]:
for i, prod in enumerate(products):
    print(f'Processing product {i+1}/{len(products)}: {prod.name}')
    nex = prod.access()

    datadict = nex.sym_block[0][0]
    data = nex.map_data(datadict['data'])
    data = np.ma.masked_invalid(data)

    az = (datadict['start_az'] + [datadict['end_az'][-1]]) * units.degrees
    rng = np.linspace(0, nex.max_range, data.shape[-1] + 1) * units.km

    lon, lat = azimuth_range_to_lat_lon(az, rng, nex.lon, nex.lat)

    fig = plt.figure(figsize=(9, 8))

    ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=nex.lon, central_latitude=nex.lat))

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(USCOUNTIES.with_scale('5m'), linewidth=0.4)

    mesh = ax.pcolormesh(lon, lat, data, cmap=cmap, norm=norm, transform=ccrs.PlateCarree())

    cbar = plt.colorbar(mesh, ax=ax, pad=0.02, shrink=0.9)
    cbar.set_label('Reflectivity (dBZ)')

    add_timestamp(ax, nex.metadata['prod_time'], y=0.02, high_contrast=True, pretext='')

    outfile = outdir / f'frame_{i:03d}.png'
    plt.savefig(outfile, bbox_inches='tight')
    plt.close(fig)

Processing product 1/73: TLX_N0B_2023_04_20_00_00_19
Processing product 2/73: TLX_N0B_2023_04_20_00_01_54
Processing product 3/73: TLX_N0B_2023_04_20_00_03_45
Processing product 4/73: TLX_N0B_2023_04_20_00_04_59
Processing product 5/73: TLX_N0B_2023_04_20_00_06_39
Processing product 6/73: TLX_N0B_2023_04_20_00_08_12
Processing product 7/73: TLX_N0B_2023_04_20_00_10_02
Processing product 8/73: TLX_N0B_2023_04_20_00_11_13
Processing product 9/73: TLX_N0B_2023_04_20_00_12_49
Processing product 10/73: TLX_N0B_2023_04_20_00_14_20
Processing product 11/73: TLX_N0B_2023_04_20_00_16_09
Processing product 12/73: TLX_N0B_2023_04_20_00_17_26
Processing product 13/73: TLX_N0B_2023_04_20_00_19_09
Processing product 14/73: TLX_N0B_2023_04_20_00_20_44
Processing product 15/73: TLX_N0B_2023_04_20_00_22_36
Processing product 16/73: TLX_N0B_2023_04_20_00_24_01
Processing product 17/73: TLX_N0B_2023_04_20_00_25_39
Processing product 18/73: TLX_N0B_2023_04_20_00_27_30
Processing product 19/73: TLX_N0B_202